In [ ]:
%pip install -r requirements.txt

In [ ]:
import cv2
import math
from ultralytics import YOLO

# ==========================================
# CONFIGURATION
# ==========================================
model_path = 'best.pt'
CONFIDENCE_THRESHOLD = 0.40

classNames = [
    "Bouchon Manquant",
    "Bouchon Mal Visse",
    "Etiquette Manquante",
    "Etiquette De travers",
    "Objet Etranger !!"
]

COLOR_RED = (0, 0, 255)
COLOR_GREEN = (0, 255, 0)
COLOR_BLACK = (0, 0, 0)

# ==========================================
# FONCTION : DÉTECTER LES CAMÉRAS DISPONIBLES
# ==========================================
def detect_cameras(max_tested=5):
    available_cameras = []
    print("\nRecherche des caméras disponibles...")
    
    for i in range(max_tested):
        cap = cv2.VideoCapture(i)
        if cap.isOpened():
            ret, _ = cap.read()
            if ret:
                available_cameras.append(i)
                print(f"✅ Caméra détectée : index {i}")
            cap.release()
    
    if not available_cameras:
        print("❌ Aucune caméra détectée.")
    
    return available_cameras

# ==========================================
# INITIALISATION DU MODÈLE
# ==========================================
print("Chargement du modèle YOLOv8...")
model = YOLO(model_path)

# ==========================================
# CHOIX DE LA CAMÉRA
# ==========================================
cameras = detect_cameras(max_tested=10)

if not cameras:
    print("Erreur : aucune caméra disponible.")
    exit()

print("\nCaméras disponibles :")
for cam in cameras:
    print(f"  - Caméra {cam}")

while True:
    try:
        cam_index = int(input("\n👉 Choisis l'index de la caméra à utiliser : "))
        if cam_index in cameras:
            break
        else:
            print("❌ Index invalide. Réessaie.")
    except ValueError:
        print("❌ Entrée invalide. Entre un nombre.")

# ==========================================
# DÉMARRAGE DE LA CAMÉRA CHOISIE
# ==========================================
cap = cv2.VideoCapture(cam_index)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

print(f"\n🎥 Caméra {cam_index} démarrée. Appuyez sur 'q' pour quitter.")

# ==========================================
# BOUCLE PRINCIPALE
# ==========================================
while True:
    success, img = cap.read()
    if not success:
        print("Erreur : Impossible de lire la caméra.")
        break

    results = model(img, stream=True, conf=CONFIDENCE_THRESHOLD)
    defect_found = False

    for r in results:
        for box in r.boxes:
            defect_found = True

            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = math.ceil((box.conf[0] * 100)) / 100
            cls = int(box.cls[0])

            current_class = classNames[cls] if cls < len(classNames) else "Inconnu"

            cv2.rectangle(img, (x1, y1), (x2, y2), COLOR_RED, 3)

            label = f'{current_class} {int(conf*100)}%'
            t_size = cv2.getTextSize(label, 0, 0.6, 2)[0]
            c2 = x1 + t_size[0], y1 - t_size[1] - 3

            cv2.rectangle(img, (x1, y1), c2, COLOR_RED, -1)
            cv2.putText(img, label, (x1, y1 - 2),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6,
                        (255, 255, 255), 1, cv2.LINE_AA)

    # ==========================================
    # STATUT GLOBAL
    # ==========================================
    cv2.rectangle(img, (0, 0), (1280, 80), COLOR_BLACK, -1)

    if defect_found:
        status_text = "NON CONFORME X"
        status_color = COLOR_RED
    else:
        status_text = "CONFORME (OK)"
        status_color = COLOR_GREEN

    cv2.putText(img, status_text, (50, 55),
                cv2.FONT_HERSHEY_SIMPLEX, 1.5,
                status_color, 3)

    cv2.imshow('Controle Qualite - YOLOv8', img)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# ==========================================
# NETTOYAGE
# ==========================================
cap.release()
cv2.destroyAllWindows()